# 01 — Exploration des données collectées

Vérifie que le pipeline Mastodon → Kafka → Spark → PostgreSQL alimente bien la base.

In [9]:
import sys, json
sys.path.insert(0, "/home/jovyan/work")
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from common import config

engine = create_engine(config.SQLALCHEMY_URL)
q = lambda sql: pd.read_sql(sql, engine)

# Palette : positif = bleu, négatif = orange (paire distinguable par les daltoniens)
COLORS = {"positive": "#2a78d6", "negative": "#eb6834"}
sns.set_theme(style="whitegrid", rc={"axes.spines.top": False, "axes.spines.right": False,
                                     "grid.color": "#e6e6e6", "axes.titleweight": "bold"})
FIG_DIR = config.OUTPUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()


## Volume et période couverte

In [10]:
q("""SELECT COUNT(*) AS nb_toots, MIN(created_at) AS debut, MAX(created_at) AS fin,
          COUNT(DISTINCT account) AS nb_comptes FROM toots""")

,nb_toots,debut,fin,nb_comptes
0,445,2026-10-09 10:45:00,2026-10-10 19:08:41,247


## Échantillon de toots enrichis

In [11]:
q("SELECT created_at, account, clean_text, hashtags, sentiment, ROUND(score_pos::numeric, 3) AS score_pos FROM toots ORDER BY created_at DESC LIMIT 10")

,created_at,account,clean_text,hashtags,sentiment,score_pos
0,2026-10-10 19:08:41,LevZadov@kolektiva.social,peterthiel ai peter thiel you know who else wa...,"peterthiel,ai",positive,0.991
1,2026-10-10 19:08:40,Maxtory9999,episode of the spyro the dragon playthrough is...,"spyrothedragon,spyro,insomniacgames,playstatio...",positive,0.988
2,2026-10-10 19:08:20,techwire@gamefan.net,free meals and other perks from big pharma rea...,"tech,technology,technews,ai,gadgets,software,c...",positive,0.987
3,2026-10-10 19:08:17,MillMessenger@journa.host,pope says death penalty inadmissible as us pla...,"news,politics,geopolitics,peace,microstateofpe...",positive,1.000
4,2026-10-10 19:08:14,60Clemens,german cemetery art artwork contemporaryart fi...,"art,artwork,contemporaryart,fineart,modernart,...",positive,0.635
5,2026-10-10 19:08:07,TejaKrasek@mathstodon.xyz,intergalactic love my today's artwork to wish ...,"galaxy,science,computer,love,digital,fractal,m...",positive,0.846
6,2026-10-10 19:08:03,yayafa@jforo.com,google chrome gb ai yayafa com p agenticai ai ...,"agenticai,ai,artificialgeneralintelligence,art...",positive,0.942
7,2026-10-10 19:07:47,LisaWarnerLisaLuv,dear beloved friends across the world some won...,"music,saturday,theme,love,one,another",positive,1.000
8,2026-10-10 19:07:46,AGT@mastodon.scot,at the risk of being argumentative i totally d...,"uspol,volodymrzelensky,president,skynews,news,...",negative,0.309
9,2026-10-10 19:07:20,rte@nuacht.live,united rugby championship leinster v cardiff u...,"sportsnews,news",positive,0.769


## Toots les plus positifs / négatifs (selon le modèle)

In [4]:
display(q("SELECT clean_text, score_pos FROM toots ORDER BY score_pos DESC LIMIT 5"))
display(q("SELECT clean_text, score_pos FROM toots ORDER BY score_pos ASC LIMIT 5"))

,clean_text,score_pos


,clean_text,score_pos


## Agrégats streaming (fenêtres de 5 min)

In [12]:
q("SELECT * FROM sentiment_window ORDER BY window_start DESC LIMIT 10")

,window_start,window_end,sentiment,nb_toots,avg_score
0,2026-10-10 18:50:00,2026-10-10 18:55:00,positive,11,0.874682
1,2026-10-10 18:50:00,2026-10-10 18:55:00,negative,6,0.255436
2,2026-10-10 18:45:00,2026-10-10 18:50:00,negative,8,0.169233
3,2026-10-10 18:45:00,2026-10-10 18:50:00,positive,12,0.920226
4,2026-10-10 18:40:00,2026-10-10 18:45:00,negative,5,0.113716
5,2026-10-10 18:40:00,2026-10-10 18:45:00,positive,11,0.863751
6,2026-10-10 18:35:00,2026-10-10 18:40:00,positive,6,0.879927
7,2026-10-10 18:35:00,2026-10-10 18:40:00,negative,6,0.276307
8,2026-10-10 18:30:00,2026-10-10 18:35:00,negative,11,0.149755
9,2026-10-10 18:30:00,2026-10-10 18:35:00,positive,19,0.863500


In [13]:
q("SELECT hashtag, SUM(nb_occurrences) AS total FROM hashtag_window GROUP BY 1 ORDER BY 2 DESC LIMIT 10")

,hashtag,total
0,ai,90.0
1,news,78.0
2,software,51.0
3,tech,44.0
4,gaming,41.0
5,music,37.0
6,science,36.0
7,technology,36.0
8,climate,33.0
9,photography,33.0


## Même lecture avec Spark (JDBC + Spark SQL)

In [ ]:
spark = config.get_spark("notebook-exploration", postgres=True, ui_port=4042)
toots = spark.read.jdbc(config.JDBC_URL, "toots", properties=config.JDBC_PROPS)
toots.createOrReplaceTempView("toots")
spark.sql("""
    SELECT hour(created_at) AS heure, sentiment, COUNT(*) AS nb
    FROM toots GROUP BY 1, 2 ORDER BY 1, 2
""").show(48)

+-----+---------+---+
|heure|sentiment| nb|
+-----+---------+---+
|    5| positive|  1|
|    6| positive|  1|
|    7| negative|  2|
|    8| positive|  1|
|   10| negative|  8|
|   11| negative|  3|
|   11| positive|  2|
|   12| positive|  5|
|   13| negative|  6|
|   13| positive|  5|
|   14| negative|  2|
|   14| positive|  4|
|   15| negative|  3|
|   15| positive| 14|
|   16| negative| 10|
|   16| positive| 21|
|   17| negative| 23|
|   17| positive| 46|
|   18| negative| 74|
|   18| positive|155|
|   19| negative| 17|
|   19| positive| 42|
+-----+---------+---+



In [8]:
spark.stop()